# Modélisation formelle du Problème de Tournée (TSPTW)

## 1. Contexte et enjeux (ADEME)

Depuis les engagements mondiaux des années 90 (Protocole de Kyoto) et face à l'urgence climatique actuelle, la réduction des émissions de gaz à effet de serre est devenue une priorité absolue. Les collectivités territoriales, accompagnées par des organismes comme l'**ADEME** (Agence de l'Environnement et de la Maîtrise de l'Énergie), cherchent activement de nouvelles solutions de mobilité intelligente.

Notre structure, **CesiCDP**, répond à cet appel à manifestation d'intérêt. L'enjeu de notre étude est de démontrer comment l'optimisation algorithmique peut réduire drastiquement l'empreinte carbone des activités de logistique (livraison, ramassage).

Pour coller au plus près de la réalité du terrain, nous avons choisi de modéliser un problème complexe intégrant deux contraintes majeures :
1. **Les fenêtres temporelles (Time Windows) :** Chaque client impose des heures strictes d'ouverture et de fermeture.
2. **Les restrictions de passage :** Le réseau routier subit des perturbations (travaux, Zones à Faibles Émissions) interdisant physiquement certaines routes.

Notre objectif est donc de modéliser mathématiquement ce problème, connu sous le nom de **TSPTW** (Traveling Salesperson Problem with Time Windows), avant d'en proposer une résolution algorithmique.

## 2. Définition du Graphe, des Paramètres et des Variables

### A. Le Graphe et les Ensembles
* **Le Graphe :** Nous définissons un graphe orienté $G = (V, A)$ où $V = \{0, 1, 2, \dots, n\}$ est l'ensemble des sommets.
* **Les Nœuds :** Le sommet $0$ représente le dépôt (point de départ et d'arrivée). Les sommets $\{1, \dots, n\}$ représentent les clients à livrer.
* **Les Arêtes :** $A$ représente l'ensemble des routes autorisées. Si une route est fermée (travaux ou zone restreinte), l'arête correspondante n'existe tout simplement pas dans $A$.

### B. Les Paramètres (Données d'entrée)
* $c_{ij}$ : le coût (ou la distance) pour aller de la ville $i$ à la ville $j$.
* $t_{ij}$ : le temps de trajet estimé pour aller de la ville $i$ à la ville $j$.
* $[e_i, l_i]$ : la fenêtre temporelle de livraison pour le client $i$.
    * $e_i$ (early) : l'heure d'ouverture (le livreur ne peut pas livrer avant).
    * $l_i$ (late) : l'heure de fermeture (le livreur ne doit absolument pas arriver après).

### C. Les Variables de Décision (Ce que l'algorithme doit trouver)
* $x_{ij} \in \{0, 1\}$ : variable binaire valant $1$ si notre véhicule emprunte directement la route allant de $i$ à $j$, et $0$ sinon.
* $T_i \ge 0$ : variable continue qui enregistre l'heure exacte de début de service du véhicule à la ville $i$.

## 3. Modélisation Mathématique

### A. La Fonction Objectif
L'objectif de notre modèle est de minimiser le coût total de la tournée (distance totale ou carburant consommé).
$$\text{Min} \sum_{i \in V} \sum_{j \in V} c_{ij} x_{ij}$$

Nous calculons la somme des coûts $c_{ij}$ de tous les trajets possibles, multipliée par notre variable de décision $x_{ij}$. Puisque $x_{ij}$ vaut $1$ si la route est empruntée et $0$ sinon, l'équation ne calculera que le coût des routes réellement utilisées par le véhicule. À noter que si une route est interdite (zone à faibles émissions, travaux), son coût $c_{ij}$ est défini à $+\infty$, ce qui empêchera mathématiquement toute solution optimale de l'emprunter.

### B. Contraintes de Parcours
Chaque client doit être visité exactement une fois.
Le véhicule doit arriver une fois chez le client $j$ :
$$\sum_{i \in V, i \neq j} x_{ij} = 1 \quad \forall j \in \{1, \dots, n\}$$

Le véhicule doit repartir une fois du client $i$ :
$$\sum_{j \in V, j \neq i} x_{ij} = 1 \quad \forall i \in \{1, \dots, n\}$$

Le véhicule doit quitter le dépôt ($0$) et y revenir à la fin de sa tournée :
$$\sum_{j=1}^{n} x_{0j} = 1 \quad \text{et} \quad \sum_{i=1}^{n} x_{i0} = 1$$

### C. Contraintes de Fenêtres Temporelles et Sous-tournées
La gestion du temps relie nos variables de parcours $x_{ij}$ à nos variables de temps $T_i$ grâce à la méthode du "Big-M" (où $M$ est une constante suffisamment grande). Si le véhicule se déplace de $i$ à $j$, l'heure d'arrivée $T_j$ doit être cohérente avec le temps de trajet :
$$T_j \ge T_i + t_{ij} - M(1 - x_{ij}) \quad \forall i \in V, \forall j \in \{1, \dots, n\}, i \neq j$$
*Note : Cette inéquation permet également d'éliminer les sous-tournées illogiques.*

Si le véhicule ne va pas de $i$ à $j$ ($x_{ij} = 0$), le terme $-M$ rend l'inégalité triviale et désactive la contrainte. En revanche, si le véhicule emprunte bien cette route ($x_{ij} = 1$), l'équation devient $T_i + t_{ij} \le T_j$. L'heure d'arrivée $T_j$ intègre donc bien le temps de trajet $t_{ij}$.
Cette contrainte est efficace car elle empêche également la formation de "sous-tours" (le véhicule qui tournerait en rond sur 3 villes sans passer par le dépôt). Le temps $T$ ne pouvant qu'augmenter, il est mathématiquement impossible de créer une boucle fermée infinie.


Enfin, le service doit impérativement s'effectuer dans la fenêtre temporelle du client :
$$e_i \le T_i \le l_i \quad \forall i \in \{1, \dots, n\}$$

## 4. Domaine de définition des variables

Enfin, nous figeons la nature mathématique de nos variables de décision.

$x_{ij} \in \{0, 1\} \quad \forall i, j \in V$
$T_i \ge 0 \quad \forall i \in V$

Les trajets sont des choix binaires (OUI ou NON), et le temps est une valeur continue strictement positive.